# Analisis Produksi Padi (Machine Learning)

Notebook ini berisi contoh implementasi Machine Learning sederhana menggunakan dataset `data.csv`.

**Algoritma yang digunakan:**
1. **K-Means Clustering (Unsupervised)**: Mengelompokkan provinsi berdasarkan pola produksi.
2. **Linear Regression (Supervised)**: Memprediksi total tahunan berdasarkan data Triwulan 1.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# Set style visualisasi
sns.set(style="whitegrid")

## 1. Load & Preprocessing Data

In [ ]:
# Load Data
file_path = 'data.csv'
df = pd.read_csv(file_path)

# Tampilkan data awal
df.head()

In [ ]:
# Cleaning Data
# Buang baris 'INDONESIA' (Agregat) dan baris kosong
df_clean = df[df['Provinsi'] != 'INDONESIA'].copy()
df_clean = df_clean.dropna()

# Cek info data
df_clean.info()

## 2. K-Means Clustering
Mengelompokkan provinsi menjadi 3 cluster berdasarkan pola produksi bulanan.

In [ ]:
features = ['Januari', 'Februari', 'Maret', 'April', 'Mei', 'Juni', 
            'Juli', 'Agustus', 'September', 'Oktober', 'November', 'Desember']

X = df_clean[features]

# Standarisasi Data (Wajib buat K-Means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fitting K-Means dengan 3 Cluster
kmeans = KMeans(n_clusters=3, random_state=42)
df_clean['Cluster'] = kmeans.fit_predict(X_scaled)

# Tampilkan hasil clustering
print("Hasil Clustering per Provinsi:")
df_clean[['Provinsi', 'Tahunan', 'Cluster']].sort_values('Cluster')

### Analisis Cluster

In [ ]:
# Rata-rata produksi per Cluster
cluster_analysis = df_clean.groupby('Cluster')[['Tahunan']].mean()
cluster_analysis.sort_values('Tahunan', ascending=False)

## 3. Linear Regression
Memprediksi total panen **Tahunan** hanya menggunakan data **Triwulan 1 (Jan-Mar)**.

In [ ]:
X_reg = df_clean[['Januari', 'Februari', 'Maret']]
y_reg = df_clean['Tahunan']

# Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Model Training
model = LinearRegression()
model.fit(X_train, y_train)

# Prediksi
y_pred = model.predict(X_test)

# Evaluasi
score = r2_score(y_test, y_pred)
print(f"R2 Score (Akurasi): {score:.2f}")

# Komparasi
comparison = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
comparison

In [ ]:
# Plot Hasil Regresi
plt.figure(figsize=(10, 6))
sns.regplot(x=y_test, y=y_pred, scatter_kws={'alpha':0.6}, line_kws={'color':'red'})
plt.xlabel('Actual Tahunan')
plt.ylabel('Predicted Tahunan')
plt.title('Actual vs Predicted (Linear Regression)')
plt.show()